In [34]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import (
    Adam,
    SGD,
    RMSprop
)
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    GlobalAveragePooling2D
)
from tensorflow.keras.applications import (
    MobileNetV2,
    EfficientNetB0,
    ResNet50,
    DenseNet121
)
import keras_tuner as kt
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

In [35]:
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

In [36]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
train_df["label"] = encoder.fit_transform(train_df["disease"])
valid_df["label"] = encoder.transform(valid_df["disease"])
test_df["label"] = encoder.transform(test_df["disease"])

In [37]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 32

In [38]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [39]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = tf.clip_by_value(
        image,
        0,
        255
    )
    image = image / 255.0
    image = data_augmentation(image)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

In [40]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [41]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [42]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [43]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
print(weights)
class_weights = dict(zip(np.unique(train_df["label"]), weights))
print(class_weights)

[ 4.37305053  2.78174603  1.30224782 12.3633157   0.21338772  1.2855309
 10.11544012]
{np.int64(0): np.float64(4.37305053025577), np.int64(1): np.float64(2.7817460317460316), np.int64(2): np.float64(1.3022478172023035), np.int64(3): np.float64(12.36331569664903), np.int64(4): np.float64(0.21338772031292808), np.int64(5): np.float64(1.285530900421786), np.int64(6): np.float64(10.115440115440116)}


In [44]:
for images, labels in train_ds.take(1):
    print(
        tf.reduce_min(images).numpy()
    )
    print(
        tf.reduce_max(images).numpy()
    )

0.0
1.0


In [45]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [46]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [47]:
base_model = DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False
densenet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
])
densenet.compile(
    optimizer="SGD",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [48]:
history_dense = densenet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/5


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 209s 928ms/step - accuracy: 0.4295 - loss: 1.6054 - val_accuracy: 0.4481 - val_loss: 1.4773 - learning_rate: 0.0100
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 193s 871ms/step - accuracy: 0.5270 - loss: 1.3601 - val_accuracy: 0.4900 - val_loss: 1.3214 - learning_rate: 0.0100
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 194s 873ms/step - accuracy: 0.5599 - loss: 1.2618 - val_accuracy: 0.5659 - val_loss: 1.1787 - learning_rate: 0.0100
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 196s 883ms/step - accuracy: 0.5830 - loss: 1.2066 - val_accuracy: 0.6778 - val_loss: 0.9676 - learning_rate: 0.0100
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 194s 873ms/step - accuracy: 0.5890 - loss: 1.1810 - val_accuracy: 0.6272 - val_loss: 1.0774 - learning_rate: 0.0100
Restoring model weights from the end of the best epoch: 4.


In [49]:
train_loss, lr_train_acc_dense = densenet.evaluate(train_ds)
valid_loss, lr_valid_acc_dense = densenet.evaluate(valid_ds)
test_loss, lr_test_acc_dense = densenet.evaluate(test_ds)
print(lr_train_acc_dense)
print(lr_valid_acc_dense)
print(lr_test_acc_dense)

220/220 ━━━━━━━━━━━━━━━━━━━━ 158s 710ms/step - accuracy: 0.6740 - loss: 0.9436
47/47 ━━━━━━━━━━━━━━━━━━━━ 34s 713ms/step - accuracy: 0.6704 - loss: 0.9561
47/47 ━━━━━━━━━━━━━━━━━━━━ 34s 720ms/step - accuracy: 0.6600 - loss: 0.9662
0.6740370988845825
0.6704394221305847
0.6600133180618286


In [50]:
den_results = pd.DataFrame(columns=[
    "Model",
    "Train accuracy",
    "Test accuracy",
    "Valid accuracy"
])
den_results.loc[len(den_results)] = [
    "densenet using SGD",
    lr_train_acc_dense,
    lr_test_acc_dense,
    lr_valid_acc_dense
]
den_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,densenet using SGD,0.674037,0.660013,0.670439


In [51]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [52]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [53]:
base_model = DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False
densenet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
])
densenet.compile(
    optimizer="RMSprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [54]:
history_dense = densenet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 200s 885ms/step - accuracy: 0.4662 - loss: 1.5494 - val_accuracy: 0.6138 - val_loss: 1.0801 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 194s 875ms/step - accuracy: 0.5660 - loss: 1.2904 - val_accuracy: 0.4760 - val_loss: 1.4969 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1147s 5s/step - accuracy: 0.5944 - loss: 1.1962 - val_accuracy: 0.5672 - val_loss: 1.2307 - learning_rate: 0.0010
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 193s 868ms/step - accuracy: 0.6088 - loss: 1.1348 - val_accuracy: 0.6445 - val_loss: 0.9597 - learning_rate: 0.0010
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 193s 869ms/step - accuracy: 0.6161 - loss: 1.0846 - val_accuracy: 0.7177 - val_loss: 0.7552 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 5.


In [55]:
train_loss, rm_train_acc_dense = densenet.evaluate(train_ds)
valid_loss, rm_valid_acc_dense = densenet.evaluate(valid_ds)
test_loss, rm_test_acc_dense = densenet.evaluate(test_ds)
print(rm_train_acc_dense)
print(rm_valid_acc_dense)
print(rm_test_acc_dense)

220/220 ━━━━━━━━━━━━━━━━━━━━ 167s 751ms/step - accuracy: 0.7305 - loss: 0.7077
47/47 ━━━━━━━━━━━━━━━━━━━━ 35s 742ms/step - accuracy: 0.7224 - loss: 0.7480
47/47 ━━━━━━━━━━━━━━━━━━━━ 35s 747ms/step - accuracy: 0.7066 - loss: 0.7971
0.7305278182029724
0.7223701477050781
0.7065868377685547


In [56]:
den_results.loc[len(den_results)] = [
    "densenet using RMSprop",
    rm_train_acc_dense,
    rm_test_acc_dense,
    rm_valid_acc_dense
]
den_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,densenet using SGD,0.674037,0.660013,0.670439
1,densenet using RMSprop,0.730528,0.706587,0.722370


In [57]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [58]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [59]:
base_model = DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False
densenet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
])
densenet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [60]:
history_dense = densenet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 202s 894ms/step - accuracy: 0.4676 - loss: 1.5475 - val_accuracy: 0.5686 - val_loss: 1.1575 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 195s 876ms/step - accuracy: 0.5742 - loss: 1.2566 - val_accuracy: 0.6385 - val_loss: 1.0037 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 206s 929ms/step - accuracy: 0.5899 - loss: 1.1714 - val_accuracy: 0.6838 - val_loss: 0.8636 - learning_rate: 0.0010
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 195s 877ms/step - accuracy: 0.6014 - loss: 1.1059 - val_accuracy: 0.6172 - val_loss: 1.0580 - learning_rate: 0.0010
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 195s 877ms/step - accuracy: 0.6245 - loss: 1.0634 - val_accuracy: 0.6325 - val_loss: 1.0509 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 3.


In [61]:
train_loss, adam_train_acc_dense = densenet.evaluate(train_ds)
valid_loss, adam_valid_acc_dense = densenet.evaluate(valid_ds)
test_loss, adam_test_acc_dense = densenet.evaluate(test_ds)
print(adam_train_acc_dense)
print(adam_valid_acc_dense)
print(adam_test_acc_dense)

220/220 ━━━━━━━━━━━━━━━━━━━━ 164s 736ms/step - accuracy: 0.6924 - loss: 0.8417
47/47 ━━━━━━━━━━━━━━━━━━━━ 34s 728ms/step - accuracy: 0.6871 - loss: 0.8623
47/47 ━━━━━━━━━━━━━━━━━━━━ 34s 721ms/step - accuracy: 0.6693 - loss: 0.8883
0.6924393773078918
0.687083899974823
0.6693280339241028


In [62]:
den_results.loc[len(den_results)] = [
    "densenet using adam",
    adam_train_acc_dense,
    adam_test_acc_dense,
    adam_valid_acc_dense
]
den_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,densenet using SGD,0.674037,0.660013,0.670439
1,densenet using RMSprop,0.730528,0.706587,0.722370
2,densenet using adam,0.692439,0.669328,0.687084


In [63]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 64

In [64]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [65]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = tf.clip_by_value(
        image,
        0,
        255
    )
    image = image / 255.0
    image = data_augmentation(image)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

In [66]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [67]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [68]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [69]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
print(weights)
class_weights = dict(zip(np.unique(train_df["label"]), weights))
print(class_weights)

[ 4.37305053  2.78174603  1.30224782 12.3633157   0.21338772  1.2855309
 10.11544012]
{np.int64(0): np.float64(4.37305053025577), np.int64(1): np.float64(2.7817460317460316), np.int64(2): np.float64(1.3022478172023035), np.int64(3): np.float64(12.36331569664903), np.int64(4): np.float64(0.21338772031292808), np.int64(5): np.float64(1.285530900421786), np.int64(6): np.float64(10.115440115440116)}


In [70]:
base_model = DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False
densenet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
])
densenet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [71]:
batch_size = 64
history_dense = densenet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    batch_size = batch_size
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - accuracy: 0.4307 - loss: 1.5342 - val_accuracy: 0.6565 - val_loss: 1.0317
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 204s 2s/step - accuracy: 0.5777 - loss: 1.2490 - val_accuracy: 0.6059 - val_loss: 1.1125
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 205s 2s/step - accuracy: 0.6144 - loss: 1.1348 - val_accuracy: 0.5406 - val_loss: 1.2492
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 206s 2s/step - accuracy: 0.6029 - loss: 1.1109 - val_accuracy: 0.6112 - val_loss: 1.1279
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 210s 2s/step - accuracy: 0.6251 - loss: 1.0556 - val_accuracy: 0.6119 - val_loss: 1.1108


In [72]:
train_loss, bt64_train_acc_dense = densenet.evaluate(train_ds)
valid_loss, bt64_valid_acc_dense = densenet.evaluate(valid_ds)
test_loss, bt64_test_acc_dense = densenet.evaluate(test_ds)
print(bt64_train_acc_dense)
print(bt64_valid_acc_dense)
print(bt64_test_acc_dense)

110/110 ━━━━━━━━━━━━━━━━━━━━ 167s 2s/step - accuracy: 0.6093 - loss: 1.0821
24/24 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.6052 - loss: 1.1117
24/24 ━━━━━━━━━━━━━━━━━━━━ 35s 1s/step - accuracy: 0.5875 - loss: 1.1533
0.6092724800109863
0.6051930785179138
0.5874916911125183


In [73]:
den_results.loc[len(den_results)] = [
    "densenet using batchsize 64",
    bt64_train_acc_dense,
    bt64_test_acc_dense,
    bt64_valid_acc_dense
]
den_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,densenet using SGD,0.674037,0.660013,0.670439
1,densenet using RMSprop,0.730528,0.706587,0.722370
2,densenet using adam,0.692439,0.669328,0.687084
3,densenet using batchsize 64,0.609272,0.587492,0.605193


In [74]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [75]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [76]:
base_model = densenet.layers[0]

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

densenet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_densenet_ft = densenet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 216s 2s/step - accuracy: 0.5756 - loss: 1.2121 - val_accuracy: 0.6278 - val_loss: 1.0752 - learning_rate: 1.0000e-05
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 214s 2s/step - accuracy: 0.6334 - loss: 1.0322 - val_accuracy: 0.6252 - val_loss: 1.0601 - learning_rate: 1.0000e-05
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 209s 2s/step - accuracy: 0.6429 - loss: 1.0186 - val_accuracy: 0.6225 - val_loss: 1.0769 - learning_rate: 1.0000e-05
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 211s 2s/step - accuracy: 0.6501 - loss: 0.9848 - val_accuracy: 0.6192 - val_loss: 1.1011 - learning_rate: 1.0000e-05
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6531 - loss: 0.9543
Epoch 5: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
110/110 ━━━━━━━━━━━━━━━━━━━━ 212s 2s/step - accuracy: 0.6531 - loss: 0.9543 - val_accuracy: 0.6318 - val_loss: 1.0651 - learning_rate: 1.0000e-05
Restoring model weights from the end of the best epoch: 2.


In [77]:
train_loss, fine_train_acc_dense = densenet.evaluate(train_ds)
valid_loss, fine_valid_acc_dense = densenet.evaluate(valid_ds)
test_loss, fine_test_acc_dense = densenet.evaluate(test_ds)
print(fine_train_acc_dense)
print(fine_valid_acc_dense)
print(fine_test_acc_dense)

110/110 ━━━━━━━━━━━━━━━━━━━━ 165s 1s/step - accuracy: 0.6338 - loss: 1.0312
24/24 ━━━━━━━━━━━━━━━━━━━━ 37s 2s/step - accuracy: 0.6325 - loss: 1.0675
24/24 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.6267 - loss: 1.1075
0.6338088512420654
0.6324900388717651
0.6267465353012085


In [78]:
den_results.loc[len(den_results)] = [
    "densenet using fine tunning",
    fine_train_acc_dense,
    fine_test_acc_dense,
    fine_valid_acc_dense
]
den_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,densenet using SGD,0.674037,0.660013,0.670439
1,densenet using RMSprop,0.730528,0.706587,0.722370
2,densenet using adam,0.692439,0.669328,0.687084
3,densenet using batchsize 64,0.609272,0.587492,0.605193
4,densenet using fine tunning,0.633809,0.626747,0.632490


In [81]:
import tensorflow as tf

def build_densenet(hp):
    base = tf.keras.applications.DenseNet121(
        include_top=False,
        weights="imagenet",
        input_shape=INPUT_SHAPE
    )
    # 1. Freeze the base model to speed up training dramatically
    base.trainable = False 

    model = tf.keras.Sequential([
        base,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(
            hp.Int("units", 64, 128, step=64),
            activation="relu"
        ),
        tf.keras.layers.Dropout(
            hp.Float("dropout", 0.2, 0.5, step=0.1)
        ),
        tf.keras.layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        )
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=hp.Float("learning_rate", 1e-5, 1e-3, sampling="log")
        ),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [83]:
densenet_tuner = kt.RandomSearch(
    build_densenet,
    objective="val_accuracy",
    max_trials=3,
    directory="tuning",
    project_name="densenet121"
)
densenet_tuner.search(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights
)

Reloading Tuner from tuning/densenet121/tuner0.json


In [84]:
best_densenet = densenet_tuner.get_best_models(1)[0]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 734 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [87]:
best_hps = densenet_tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'units': 64, 'dropout': 0.4, 'learning_rate': 2.770506871918118e-05}


In [89]:
base_model = DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False
densenet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(64, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
])
densenet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [90]:
history_dense = densenet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5
)

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 380s 3s/step - accuracy: 0.6662 - loss: 0.9899 - val_accuracy: 0.7124 - val_loss: 0.8190
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 375s 3s/step - accuracy: 0.7138 - loss: 0.7843 - val_accuracy: 0.7310 - val_loss: 0.7557
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 375s 3s/step - accuracy: 0.7298 - loss: 0.7397 - val_accuracy: 0.7324 - val_loss: 0.7215
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 379s 3s/step - accuracy: 0.7447 - loss: 0.6987 - val_accuracy: 0.7477 - val_loss: 0.6946
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 376s 3s/step - accuracy: 0.7448 - loss: 0.6821 - val_accuracy: 0.7497 - val_loss: 0.6850


In [91]:
train_loss, hype_train_acc_dense = densenet.evaluate(train_ds)
valid_loss, hype_valid_acc_dense = densenet.evaluate(valid_ds)
test_loss, hype_test_acc_dense = densenet.evaluate(test_ds)
print(hype_train_acc_dense)
print(hype_valid_acc_dense)
print(hype_test_acc_dense)

110/110 ━━━━━━━━━━━━━━━━━━━━ 309s 3s/step - accuracy: 0.7575 - loss: 0.6675
24/24 ━━━━━━━━━━━━━━━━━━━━ 65s 3s/step - accuracy: 0.7617 - loss: 0.6720
24/24 ━━━━━━━━━━━━━━━━━━━━ 64s 3s/step - accuracy: 0.7379 - loss: 0.7354
0.7574893236160278
0.7616511583328247
0.7378576397895813


In [92]:
den_results.loc[len(den_results)] = [
    "densenet using hyperparameter",
    hype_train_acc_dense,
    hype_test_acc_dense,
    hype_valid_acc_dense
]
den_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,densenet using SGD,0.674037,0.660013,0.670439
1,densenet using RMSprop,0.730528,0.706587,0.722370
2,densenet using adam,0.692439,0.669328,0.687084
3,densenet using batchsize 64,0.609272,0.587492,0.605193
4,densenet using fine tunning,0.633809,0.626747,0.632490
5,densenet using hyperparameter,0.757489,0.737858,0.761651


In [95]:
best_densenet.save("cnn_densenet_phase5.keras")

In [96]:
densenet.save("cnn_densenet_bestmodel.keras")